# Provident Bank Statement PDF to Excel Extraction Engine

**Purpose:** Extract structured banking data from Provident PDF statements with 100% structural fidelity.

**Features:**
- OCR processing for scanned documents
- Layout-aware section detection
- Strict JSON output formatting
- Account Summary extraction
- Transaction Detail extraction
- Excel report generation

In [1]:
# Import Required Libraries
import os
import json
import re
import requests
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import pandas as pd
import PyPDF2
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Configuration
INPUT_FOLDER = r"C:\Users\ankur\Downloads\IHC\2025 Bank Statements Part 2\2025 Bank Statements Part 2\Files\10.2025\Provident"
OUTPUT_FOLDER = r"C:\Users\ankur\Downloads\IHC\10.2025\Output_Provident"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# LM Studio API Configuration
# OCR Model: olmocr-2-7b-1025-mlx for scanned document text extraction
# Structuring Model: gpt-oss-120b-mlx for JSON data extraction
LM_STUDIO_API_KEY = "lm-studio"  # LM Studio uses this as default
LM_STUDIO_API_BASE = "http://10.216.50.171:1234/v1"
OCR_MODEL_NAME = "olmocr-2-7b-1025-mlx"  # For OCR tasks
STRUCTURING_MODEL_NAME = "gpt-oss-120b-mlx"  # For JSON extraction

print(f"Input Folder: {INPUT_FOLDER}")
print(f"Output Folder: {OUTPUT_FOLDER}")
print(f"LM Studio Endpoint: {LM_STUDIO_API_BASE}")
print(f"OCR Model: {OCR_MODEL_NAME}")
print(f"Structuring Model: {STRUCTURING_MODEL_NAME}")

Input Folder: C:\Users\ankur\Downloads\IHC\2025 Bank Statements Part 2\2025 Bank Statements Part 2\Files\10.2025\Provident
Output Folder: C:\Users\ankur\Downloads\IHC\10.2025\Output_Provident
LM Studio Endpoint: http://10.216.50.171:1234/v1
OCR Model: olmocr-2-7b-1025-mlx
Structuring Model: gpt-oss-120b-mlx


## 1. PDF Processing Functions

In [3]:
def is_scanned_pdf(pdf_path: str) -> bool:
    """
    Determine if a PDF is scanned (image-based) or system-generated (text-based).
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        True if scanned, False if system-generated
    """
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            # Check first 3 pages (or all if fewer)
            pages_to_check = min(3, len(pdf_reader.pages))
            total_text = ""
            
            for page_num in range(pages_to_check):
                page = pdf_reader.pages[page_num]
                text = page.extract_text() or ""
                total_text += text.strip()
            
            # If very little text extracted, it's likely scanned
            if len(total_text) < 100:
                return True
            
            return False
    except Exception as e:
        print(f"Error checking PDF type: {e}")
        return True  # Default to OCR if uncertain

In [4]:
def extract_text_from_system_pdf(pdf_path: str) -> str:
    """
    Extract text from a system-generated PDF.
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        Extracted text with page separators
    """
    try:
        full_text = ""
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            
            for page_num in range(len(pdf_reader.pages)):
                page = pdf_reader.pages[page_num]
                text = page.extract_text() or ""
                full_text += f"\n--- PAGE {page_num + 1} ---\n{text}\n"
        
        return full_text
    except Exception as e:
        print(f"Error extracting text from system PDF: {e}")
        return ""

In [5]:
def extract_text_from_scanned_pdf(pdf_path: str) -> str:
    """
    Extract text from a scanned PDF using OCR (Tesseract).
    Preserves layout structure and spacing.
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        OCR-extracted text with page separators
    """
    try:
        print(f"Performing OCR on: {os.path.basename(pdf_path)}")
        
        # Convert PDF to images
        images = convert_from_path(pdf_path, dpi=300)
        
        full_text = ""
        for page_num, image in enumerate(images, start=1):
            print(f"  Processing page {page_num}/{len(images)}...")
            
            # Perform OCR with layout preservation
            custom_config = r'--oem 3 --psm 6'
            text = pytesseract.image_to_string(image, config=custom_config)
            
            full_text += f"\n--- PAGE {page_num} ---\n{text}\n"
        
        print(f"OCR completed: {len(full_text)} characters extracted")
        return full_text
    except Exception as e:
        print(f"Error performing OCR: {e}")
        return ""

In [6]:
def extract_full_text_from_pdf(pdf_path: str) -> Tuple[str, bool]:
    """
    Main PDF text extraction function.
    Automatically detects if PDF is scanned or system-generated.
    
    Args:
        pdf_path: Path to the PDF file
    
    Returns:
        Tuple of (extracted_text, is_scanned)
    """
    is_scanned = is_scanned_pdf(pdf_path)
    
    if is_scanned:
        print("Detected: SCANNED PDF - Using OCR")
        text = extract_text_from_scanned_pdf(pdf_path)
    else:
        print("Detected: SYSTEM-GENERATED PDF - Using text extraction")
        text = extract_text_from_system_pdf(pdf_path)
    
    return text, is_scanned

## 2. LLM-Based Structured Extraction

In [7]:
def build_extraction_prompt(raw_text: str) -> str:
    """
    Build the comprehensive extraction prompt for the LLM.
    
    Args:
        raw_text: Raw text extracted from PDF
    
    Returns:
        Formatted prompt string
    """
    prompt = f"""You are a financial document extraction engine.

Your task is to extract structured banking data with 100% structural fidelity.

IMPORTANT RULES:
1. Process the entire document page-by-page.
2. Preserve section hierarchy exactly as it appears.
3. Detect section headers using visual layout cues (position, spacing, bold text).
4. DO NOT guess missing values.
5. DO NOT hallucinate fields.
6. DO NOT summarize.
7. Output STRICT JSON ONLY.
8. If a value is not found, return null.
9. Maintain exact numeric precision (no rounding).
10. Preserve negative signs and parentheses formatting.

SECTION DETECTION LOGIC:

1) "Account Summary" or "Balance Summary" section
   - Usually appears near top of first page
   - Contains summary level account data
   - Ends before transaction table begins

2) "Transaction Detail" or "Transaction Details" section
   - Appears after summary
   - Contains tabular data with rows
   - Columns are aligned horizontally
   - May span multiple pages

ACCOUNT SUMMARY EXTRACTION:

Extract EXACTLY:
- Account Number
- Statement Date
- Account Type
- Beginning Balance
- Ending Balance
- Current Balance (if present)

Rules:
- Account Number must match exactly as printed.
- Statement Date must be extracted in original format.
- Monetary values must preserve commas and decimals.
- Do NOT infer values from transaction totals.

TRANSACTION DETAIL EXTRACTION:

Identify column headers by visual alignment.
Typical columns include: Date, Description, Deposits (Credit), Withdrawals (Debit), Balance

Rules:
1. Extract row-by-row.
2. Maintain chronological order as shown.
3. Preserve multi-line descriptions by merging them into one string.
4. If a transaction spans multiple lines, combine correctly.
5. Deposits must always be positive.
6. Withdrawals must always be positive (do not convert to negative).
7. Running Balance must match exactly as printed.
8. Do NOT calculate missing balances.
9. If a column is blank for a row, return null.

OUTPUT FORMAT (STRICT JSON ONLY):

{{
  "Account_Summary": {{
    "Account_Number": "",
    "Statement_Date": "",
    "Account_Type": "",
    "Beginning_Balance": "",
    "Ending_Balance": "",
    "Current_Balance": ""
  }},
  "Transaction_Details": [
    {{
      "Date": "",
      "Description": "",
      "Deposits": "",
      "Withdrawals": "",
      "Balance": ""
    }}
  ]
}}

ABSOLUTE CONSTRAINTS:
- DO NOT include explanations.
- DO NOT include markdown.
- DO NOT include comments.
- DO NOT include confidence statements.
- DO NOT include text outside JSON.
- Output must start with {{ and end with }}.

=== DOCUMENT TEXT START ===
{raw_text}
=== DOCUMENT TEXT END ===

Return only valid JSON:"""
    
    return prompt

In [8]:
def call_llm_for_extraction(raw_text: str, api_key: str, api_base: str, model: str) -> Optional[Dict]:
    """
    Call the LLM API to extract structured data from raw text.
    Uses requests library for better compatibility with LM Studio.
    
    Args:
        raw_text: Raw text extracted from PDF
        api_key: API key for authentication
        api_base: Base URL for API endpoint
        model: Model name to use
    
    Returns:
        Parsed JSON dictionary or None if failed
    """
    try:
        prompt = build_extraction_prompt(raw_text)
        
        print("Calling LLM for structured extraction...")
        print(f"  Endpoint: {api_base}")
        print(f"  Model: {model}")
        print(f"  Prompt length: {len(prompt)} characters")
        
        # Prepare the request
        url = f"{api_base}/chat/completions"
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        }
        
        payload = {
            "model": model,
            "messages": [
                {
                    "role": "system",
                    "content": "You are a financial document extraction engine. Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0,
            "top_p": 1,
            "max_tokens": 16000,
            "stream": False
        }
        
        # Make the request with timeout
        print("  Sending request to LM Studio...")
        response = requests.post(url, json=payload, headers=headers, timeout=300)
        
        # Check response status
        if response.status_code != 200:
            print(f"✗ HTTP Error {response.status_code}: {response.text}")
            return None
        
        # Parse response
        response_json = response.json()
        
        if "choices" not in response_json or len(response_json["choices"]) == 0:
            print(f"✗ Invalid response structure: {response_json}")
            return None
        
        # Extract content
        content = response_json["choices"][0]["message"]["content"].strip()
        print(f"  Response received: {len(content)} characters")
        
        # Remove markdown code blocks if present
        content = re.sub(r'^```json\s*', '', content)
        content = re.sub(r'^```\s*', '', content)
        content = re.sub(r'\s*```$', '', content)
        content = content.strip()
        
        # Parse JSON
        extracted_data = json.loads(content)
        
        print("✓ LLM extraction completed successfully")
        return extracted_data
        
    except requests.exceptions.Timeout:
        print("✗ Request timed out after 300 seconds")
        return None
    except requests.exceptions.ConnectionError as e:
        print(f"✗ Connection error: {e}")
        print("  Check if LM Studio is running and the endpoint is correct")
        return None
    except requests.exceptions.RequestException as e:
        print(f"✗ Request error: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ JSON parsing error: {e}")
        print(f"  Response content (first 500 chars): {content[:500]}...")
        return None
    except KeyError as e:
        print(f"✗ Missing key in response: {e}")
        print(f"  Full response: {response_json}")
        return None
    except Exception as e:
        print(f"✗ Unexpected error: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        return None

## 3. Data Validation and Quality Checks

In [9]:
def validate_extracted_data(data: Dict) -> Tuple[bool, List[str]]:
    """
    Validate the extracted data structure and perform quality checks.
    
    Args:
        data: Extracted JSON data
    
    Returns:
        Tuple of (is_valid, list_of_warnings)
    """
    warnings = []
    is_valid = True
    
    # Check required top-level keys
    if "Account_Summary" not in data:
        warnings.append("Missing Account_Summary section")
        is_valid = False
    
    if "Transaction_Details" not in data:
        warnings.append("Missing Transaction_Details section")
        is_valid = False
    
    if not is_valid:
        return is_valid, warnings
    
    # Validate Account Summary
    summary = data["Account_Summary"]
    required_fields = ["Account_Number", "Statement_Date", "Account_Type", 
                       "Beginning_Balance", "Ending_Balance"]
    
    for field in required_fields:
        if field not in summary:
            warnings.append(f"Missing field in Account_Summary: {field}")
    
    # Validate Transaction Details
    transactions = data["Transaction_Details"]
    
    if not isinstance(transactions, list):
        warnings.append("Transaction_Details must be a list")
        is_valid = False
    elif len(transactions) == 0:
        warnings.append("No transactions found")
    else:
        # Check first transaction for required fields
        required_tx_fields = ["Date", "Description"]
        for field in required_tx_fields:
            if field not in transactions[0]:
                warnings.append(f"Missing field in transactions: {field}")
    
    # Balance validation (informational only - DO NOT modify data)
    try:
        beginning = summary.get("Beginning_Balance", "")
        ending = summary.get("Ending_Balance", "")
        
        if beginning and ending:
            # Parse monetary values
            beginning_val = float(str(beginning).replace(",", "").replace("$", ""))
            ending_val = float(str(ending).replace(",", "").replace("$", ""))
            
            # Calculate totals from transactions
            total_deposits = 0
            total_withdrawals = 0
            
            for tx in transactions:
                if tx.get("Deposits") and tx.get("Deposits") not in [None, "", "null"]:
                    total_deposits += float(str(tx["Deposits"]).replace(",", "").replace("$", ""))
                if tx.get("Withdrawals") and tx.get("Withdrawals") not in [None, "", "null"]:
                    total_withdrawals += float(str(tx["Withdrawals"]).replace(",", "").replace("$", ""))
            
            calculated_ending = beginning_val + total_deposits - total_withdrawals
            
            if abs(calculated_ending - ending_val) > 0.01:
                warnings.append(f"Balance mismatch: Beginning {beginning_val} + Deposits {total_deposits} - Withdrawals {total_withdrawals} = {calculated_ending}, but Ending Balance is {ending_val}")
    except Exception as e:
        warnings.append(f"Could not validate balance calculation: {e}")
    
    return is_valid, warnings

## 4. Excel Report Generation

In [10]:
def create_excel_report(data: Dict, output_path: str) -> bool:
    """
    Create an Excel report from extracted data.
    
    Args:
        data: Extracted JSON data
        output_path: Path to save Excel file
    
    Returns:
        True if successful, False otherwise
    """
    try:
        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            # Sheet 1: Account Summary
            summary = data["Account_Summary"]
            summary_df = pd.DataFrame([
                {"Field": k, "Value": v} for k, v in summary.items()
            ])
            summary_df.to_excel(writer, sheet_name='Account Summary', index=False)
            
            # Sheet 2: Transaction Details
            transactions = data["Transaction_Details"]
            if transactions:
                tx_df = pd.DataFrame(transactions)
                tx_df.to_excel(writer, sheet_name='Transaction Details', index=False)
            
            # Sheet 3: Raw JSON
            raw_json_df = pd.DataFrame([{"Raw JSON": json.dumps(data, indent=2)}])
            raw_json_df.to_excel(writer, sheet_name='Raw JSON', index=False)
        
        print(f"✓ Excel report created: {output_path}")
        return True
        
    except Exception as e:
        print(f"✗ Error creating Excel report: {e}")
        return False

## 5. Main Processing Pipeline

In [11]:
def process_single_pdf(pdf_path: str, output_folder: str, api_key: str, api_base: str, model: str) -> bool:
    """
    Process a single PDF file through the complete extraction pipeline.
    
    Args:
        pdf_path: Path to the PDF file
        output_folder: Folder to save outputs
        api_key: API key for LLM
        api_base: API base URL
        model: Model name
    
    Returns:
        True if successful, False otherwise
    """
    try:
        filename = os.path.basename(pdf_path)
        base_name = os.path.splitext(filename)[0]
        
        print(f"\n{'='*70}")
        print(f"Processing: {filename}")
        print(f"{'='*70}")
        
        # Step 1: Extract text from PDF
        print("\n[1/5] Extracting text from PDF...")
        raw_text, is_scanned = extract_full_text_from_pdf(pdf_path)
        
        if not raw_text:
            print("✗ Failed to extract text from PDF")
            return False
        
        print(f"✓ Extracted {len(raw_text)} characters")
        
        # Step 2: Call LLM for structured extraction
        print("\n[2/5] Calling LLM for structured extraction...")
        extracted_data = call_llm_for_extraction(raw_text, api_key, api_base, model)
        
        if not extracted_data:
            print("✗ Failed to extract structured data")
            return False
        
        # Step 3: Validate extracted data
        print("\n[3/5] Validating extracted data...")
        is_valid, warnings = validate_extracted_data(extracted_data)
        
        if warnings:
            print("⚠ Validation warnings:")
            for warning in warnings:
                print(f"  - {warning}")
        else:
            print("✓ Data validation passed")
        
        if not is_valid:
            print("✗ Critical validation errors found")
            return False
        
        # Step 4: Save JSON output
        print("\n[4/5] Saving JSON output...")
        json_output_path = os.path.join(output_folder, f"{base_name}_extracted.json")
        with open(json_output_path, 'w', encoding='utf-8') as f:
            json.dump(extracted_data, f, indent=2, ensure_ascii=False)
        print(f"✓ JSON saved: {json_output_path}")
        
        # Step 5: Create Excel report
        print("\n[5/5] Creating Excel report...")
        excel_output_path = os.path.join(output_folder, f"{base_name}_report.xlsx")
        success = create_excel_report(extracted_data, excel_output_path)
        
        if success:
            print("\n" + "="*70)
            print(f"✓ PROCESSING COMPLETE: {filename}")
            print("="*70)
            return True
        else:
            return False
        
    except Exception as e:
        print(f"\n✗ Error processing {pdf_path}: {e}")
        import traceback
        traceback.print_exc()
        return False

In [12]:
def process_all_pdfs(input_folder: str, output_folder: str, api_key: str, api_base: str, model: str):
    """
    Process all PDF files in the input folder.
    
    Args:
        input_folder: Folder containing PDF files
        output_folder: Folder to save outputs
        api_key: API key for LLM
        api_base: API base URL
        model: Model name
    """
    # Find all PDF files
    pdf_files = list(Path(input_folder).glob("*.pdf"))
    
    if not pdf_files:
        print(f"No PDF files found in {input_folder}")
        return
    
    print(f"\nFound {len(pdf_files)} PDF file(s) to process")
    print("="*70)
    
    results = []
    for pdf_path in pdf_files:
        success = process_single_pdf(
            str(pdf_path),
            output_folder,
            api_key,
            api_base,
            model
        )
        results.append((pdf_path.name, success))
    
    # Summary
    print("\n" + "="*70)
    print("PROCESSING SUMMARY")
    print("="*70)
    
    successful = sum(1 for _, success in results if success)
    failed = len(results) - successful
    
    print(f"Total files: {len(results)}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print("\nDetailed results:")
    
    for filename, success in results:
        status = "✓" if success else "✗"
        print(f"  {status} {filename}")
    
    print("="*70)

## 6. Execute Processing

In [13]:
# Execute the processing pipeline
if __name__ == "__main__":
    # Process all PDFs using LM Studio endpoint
    process_all_pdfs(
        input_folder=INPUT_FOLDER,
        output_folder=OUTPUT_FOLDER,
        api_key=LM_STUDIO_API_KEY,
        api_base=LM_STUDIO_API_BASE,
        model=STRUCTURING_MODEL_NAME
    )


Found 9 PDF file(s) to process

Processing: 10.1-10.31.2025-Debt PaymentsExport-31102025.pdf

[1/5] Extracting text from PDF...
Detected: SYSTEM-GENERATED PDF - Using text extraction
✓ Extracted 1923 characters

[2/5] Calling LLM for structured extraction...
Calling LLM for structured extraction...
  Endpoint: http://10.216.50.171:1234/v1
  Model: gpt-oss-120b-mlx
  Prompt length: 4577 characters
  Sending request to LM Studio...
  Response received: 1139 characters
✓ LLM extraction completed successfully

[3/5] Validating extracted data...
✓ Data validation passed

[4/5] Saving JSON output...
✓ JSON saved: C:\Users\ankur\Downloads\IHC\10.2025\Output_Provident\10.1-10.31.2025-Debt PaymentsExport-31102025_extracted.json

[5/5] Creating Excel report...
✓ Excel report created: C:\Users\ankur\Downloads\IHC\10.2025\Output_Provident\10.1-10.31.2025-Debt PaymentsExport-31102025_report.xlsx

✓ PROCESSING COMPLETE: 10.1-10.31.2025-Debt PaymentsExport-31102025.pdf

Processing: 10.2025 Dunedin D

## 7. Test with Single File (Optional)

In [14]:
# Test with a single PDF file
# Uncomment and modify the path below to test with a specific file

# test_pdf_path = r"c:\Users\Shreeya.Nitturkar\Downloads\IHC\AI MODEL IHC\Input_Provident\sample.pdf"
# if os.path.exists(test_pdf_path):
#     process_single_pdf(
#         pdf_path=test_pdf_path,
#         output_folder=OUTPUT_FOLDER,
#         api_key=LM_STUDIO_API_KEY,
#         api_base=LM_STUDIO_API_BASE,
#         model=STRUCTURING_MODEL_NAME
#     )
# else:
#     print(f"Test file not found: {test_pdf_path}")